# Vraksh 
A library consumes documents and answer questions based on that document

## Canvas Designer Library

#### Console Print Class

In [1]:
## This takes care of standard output
from rich.console import Console
from rich.table import Table
from rich.pretty import Pretty

console =  Console()

def table_create(table_name, *columns):
    table = Table(title=table_name, show_header= True)

    for col in columns:
        table.add_column(col)
    
    return table


def table_print(table):
    console.print(table)

#### Action Designer Class

In [144]:
# VR-4565
class ActionParser:

    def __init__(self, scene):
        self.scene = scene
        self.result = []


    def parse(self):

        for token in self.scene:

            if token.pos_.casefold() == "VERB".casefold() and token.dep_.casefold() == "ROOT".casefold():
                self.result.append(token)
        
        return self.result



### Testing 
import spacy
from spacy import displacy

nlp = spacy.load("en_core_web_sm")
scene = nlp("Sunday, I eat fruits and drink water at 12:00 midnight")
action_parser = ActionParser(scene)

assert action_parser.parse()[0].text, "eat"

#### Action Receiver Designer Class

In [145]:
# VR-1212
class ActionReceiverParser:

    def __init__(self, token_action, token_action_doer):
        self.token_action = token_action
        self.token_action_doer = token_action_doer
        self.result = []


    def parse(self):

        is_doer = False

        for token in self.token_action.children:
            ## First check if this action is associated with this action_doer 
            if(token.dep_.casefold() == "nsubj"):
                is_doer = True
                break


        if is_doer:

            for token in self.token_action.children:

                ## Now check if this action has any direct object which is non living
                if (token.dep_.casefold() == "dobj"):
                    self.result.append(token)

                ## We have to handle the cases when object is not direct object and living object
                if (token.dep_.casefold() == "prep"):
                    self._parse_prep(token)


        return self.result


    # In this method we will evaluate if object could be the prep object
    def _parse_prep(self, token):

        if (token.text.casefold() == "with") or (token.text.casefold() == "to"):

            prep_obj_list = [child for child in token.children if child.dep_.casefold() == "pobj"]
            if list(prep_obj_list):
                self.result.append(prep_obj_list[0])


### Testing 
import spacy
from spacy import displacy

nlp = spacy.load("en_core_web_sm")

scene = nlp("I am playing with my calculator which is placed on the table")
action_receiver_parser = ActionReceiverParser(scene[2], scene[0])
action_receiver_parser.parse()
assert action_receiver_parser.result[0].text == "calculator", "FAILED" 

scene = nlp("I am going to school")
action_receiver_parser = ActionReceiverParser(scene[2], scene[0])
action_receiver_parser.parse()
assert action_receiver_parser.result[0].text == "school", "FAILED" 


scene = nlp("I am playing cricket in the ground")
action_receiver_parser = ActionReceiverParser(scene[2], scene[0])
action_receiver_parser.parse()
assert action_receiver_parser.result[0].text == "cricket", "FAILED" 

#### Action Doer Designer Class

In [146]:
# VR-4323
class ActionDoerParser:

    def __init__(self, token_verb):
        self.token_verb = token_verb
        self.result = []


    def parse(self):

        for token in self.token_verb.children:

            if token.dep_.casefold() == "nsubj".casefold():
                self.result.append(token)

        return self.result
    

### Testing 
import spacy
from spacy import displacy

nlp = spacy.load("en_core_web_sm")


scene = nlp("Amit eats fruits")

action_doer_parser = ActionDoerParser(scene[1])
assert action_doer_parser.parse()[0].text, "Amit"


#### Action Time Designer Class

In [147]:
# VR-3434
class ActionTimeParser:

    def __init__(self, token_verb):
        self.token_verb = token_verb
        self.return_day = []
        self.return_time = []
        self.result = []


    def parse(self):
        self._parse(self.token_verb)
        self.result = self.return_day + self.return_time
        return self.result

    def _parse(self, token):

        # It means I am processing the target verb token
        if (token.text.casefold() == self.token_verb.text.casefold()):

            # Now loop through all children of this main verb and pass each of them 
            for child in token.children:
                self._parse(child)


        ## Now check if this is the npadvmod case 
        if (token.dep_.casefold() == "npadvmod" and token.ent_type_.casefold() == "DATE".casefold()):
            
            self.return_day.append(token)

            # Now move further on its children
            for child in token.children:
                self._parse(child)

        ## Now check if this is the npadvmod case 
        if (token.dep_.casefold() == "npadvmod" and token.ent_type_.casefold() == "TIME".casefold()):
                    
            self.return_time.append(token)
        
            # Now move further on its children
            for child in token.children:
                self._parse(child)


        ## Now check prep at case  
        if(token.dep_ == "prep" and (token.text.casefold() == "at".casefold() or token.text.casefold() == "around".casefold() or token.text.casefold() == "on".casefold())):

            ## Process further
            for child in token.children:
                self._parse(child)


        
        ## Below is the case of the extracting time (AM/PM)
        if((token.head.text.casefold() == "at".casefold() or token.head.text.casefold() == "around".casefold()) and token.dep_.casefold() == "pobj" and token.ent_type_ == "TIME"):

            ## Process further
            ## Handle cases like 
            ## at 12:00 AM , at midnight
            if (token.pos_.casefold() == "NOUN".casefold() or token.pos_.casefold() == "PROPN".casefold()):

                if list(token.children):

                    ## at 12:00 AM
                    for child in token.children:
                        self._parse(child)
                else:

                    ## at midnight
                    self.return_time.append(token)

            ## handle cases at 12:00 
            elif (token.pos_.casefold() == "NUM".casefold()):
                self.return_time.append(token)
        

        ## Below is the case of the extracting time (AM/PM)
        if((token.head.text.casefold() == "on".casefold() or token.head.text.casefold() == "around".casefold()) and token.dep_.casefold() == "pobj" and token.ent_type_ == "DATE"):

            if (token.pos_.casefold() == "NOUN".casefold() or token.pos_.casefold() == "PROPN".casefold()):

                if list(token.children):
                
                    ## 
                    for child in token.children:
                        self._parse(child)
                else:
                
                    ## on Monday
                    self.return_day.append(token)


        ## Below is the case of extracting time 
        if(token.head.ent_type_.casefold() == "TIME".casefold() and token.dep_.casefold() == "nummod".casefold() and token.ent_type_.casefold() == "TIME".casefold()):

            ## Process further
            self.return_time.append([token, token.head])




### Testing 
import spacy
from spacy import displacy

nlp = spacy.load("en_core_web_sm")
scene = nlp("Sunday, I eat fruits at 12:00 AM")

action_time_parser = ActionTimeParser(scene[3])
action_time_parser.parse()
assert ",".join( time.text for time in action_time_parser.return_time[0]), "12:00, AM"
assert action_time_parser.return_day[0].text, "Sunday"

### Scene Class

In [ ]:
from rich import print

import spacy
from spacy import displacy
from spacy.tokens import Token


class Scene:

    def __init__(self, scene_str):
        self.scene = nlp(scene_str)
        Token.set_extension("_to_element", method=self._to_element, force=True)
    
    
    def get_action(self):
        action_parser = ActionParser(self.scene)
        return action_parser.parse()


    def get_action_doer(self, action):
        action_doer_parser = ActionDoerParser(action)
        return action_doer_parser.parse()

    def get_action_time(self, action):
        action_time_parser = ActionTimeParser(action)
        return action_time_parser.parse()


    def get_action_receiver(self, action, action_doer):
        action_receiver_parser = ActionReceiverParser(action, action_doer)
        return action_receiver_parser.parse()

    




## Timeline Class

In [ ]:
from uuid import UUID

from rich import print
from tinydb import TinyDB, Query
from rich import print_json
import json


class Timeline:

    def __init__(self):
        self.db = TinyDB('my_db.json')
        self.id = 0
        self.db.truncate()

    def get_unique_id(self):
        self.id = self.id + 1
        return self.id

    
    def add_action(self, unique_id, action: Element):

        query = Query()
        result = self.db.search(query.id == unique_id)
        if not result:
            self.db.insert({"id" : unique_id, "action" : action._to_dict()})
        else:
            query = Query()
            self.db.update({"action" : action._to_dict()}, query.id == unique_id)



    def add_action_doer(self, unique_id, action :Element, action_doer :Element):

        query = Query()
        result = self.db.search(query.id == unique_id)
        if not result:
            self.db.insert({"id" : unique_id, "action" : action._to_dict(), "action_doer" : action_doer._to_dict()})
        else:
            self.db.update({"action_doer" : action_doer._to_dict()}, query.id == unique_id)


    def add_action_time(self, unique_id, action :Element, action_time :Element):

        query = Query()
        result = self.db.search(query.id == unique_id)
        if not result:
            if isinstance(action_time, list):
                self.db.insert({"id" : unique_id, "action" : action._to_dict(), "action_time" : [ x._to_dict() for x in action_time ]})
            else:
                self.db.insert({"id" : unique_id, "action" : action._to_dict(), "action_time" :  action_time._to_dict()})
        else:
            query = Query()
            if isinstance(action_time, list):
                self.db.update({"id" : unique_id, "action_time" : [ x._to_dict() for x in action_time ]}, query.id == unique_id)
            else:
                self.db.update({"id" : unique_id, "action_time" :  action_time._to_dict()}, query.id == unique_id)



    def add_action_receiver(self, unique_id, action :Element, action_doer:Element, action_receiver :Element):

        query = Query()
        result = self.db.search(query.id == unique_id)
        if not result:
            self.db.insert({"id" : unique_id, "action" : action._to_dict(), "action_receiver" : action_receiver._to_dict()})
        else:
            query = Query()
            self.db.update({"action_receiver" : action_receiver._to_dict()}, query.id == unique_id)

            

    def add_action_doer_location(self, unique_id, action, action_doer, location):
        pass


    def add_action_object_location(self, unique_id, action , action_object, location):
        pass


    def clear(self):
        self.db.truncate()


    def render(self):
        pretty_json = json.dumps(self.db.all(), indent=4)
        print(pretty_json)


    def query(self, query):

        if isinstance(query, QuerySubject):
            return self.query_subject(query)


    def query_subject(self, query_subject):

        ## Find subject which are matching action name 
        query = Query()
        result = self.db.search(query.action.text == query_subject.action.text)
        return result



In [150]:
class QuerySubject:

    def action(self, action):
        self.action = action
        return self

    # action_time = None, action_location = None, action_receiver = None):
    def action_time(self, action_time):
        self.action_time = action_time
        return self

    def action_location(self, action_location):
        self.action_location = action_location
        return self

    def action_receiver(self, action_receiver):
        self.action_receiver = action_receiver
        return self

## Canvas Reader Library

### Canvas Reader Class


# Main

In [158]:
from rich import print

import spacy
from spacy import displacy
from spacy.tokens import Token

nlp = spacy.load("en_core_web_trf")

timeline = Timeline()

scene = Scene("Amit wake up at 09:00 AM today")
scene.add_scene_to_timeline(timeline)

scene2 = Scene("Rahul drank water at 10:00 AM")
scene2.add_scene_to_timeline(timeline)

scene3 = Scene("Deepak took bath at 10:30 AM")
scene3.add_scene_to_timeline(timeline)

scene4 = Scene("Praveen started my office at 11:00 AM")
scene4.add_scene_to_timeline(timeline)

# timeline.render()

# ### ----------------------------------------------------------

scene_reader = Scene("who took bath at 10:30 AM")


query_subject = QuerySubject()
query_subject.action(scene_reader.get_action()[0])

result = timeline.query(query=query_subject)

for r in result:
    print(r)


TypeError: Object of type Element is not JSON serializable